[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/poncelet/poncelet.ipynb)

# Poncelet's Porism

This example draws polygons inscribed in one conic and circumscribed about another. From a point on an outer ellipse, draw a tangent to an inner conic, follow it to where it meets the ellipse again, take the other tangent from there, and go on. Poncelet's porism: if the path closes after some number of sides from one starting point, it closes after that number from every starting point, and if it misses from one, it misses from all.

Points are positions, antivectors, and the outer ellipse is a map from a point to its polar line, `Plane <- Point`, zero on its points. The sides of the path are chords, joins of two points, and the inner conic is held by the chords that touch it: it sends a chord to its pole, the meet of the polars of the chord's points, `Pole <- Chord`, and a chord touches it where it passes through its own pole. Everything about one conic has a twin about the other: five points fix a conic through them and five chords a conic they touch; a chord from a point meets a conic once more, and through a point pass two tangents. Knowing one of the two, the other follows without a square root, by the same formula, once along directions to the ellipse and once along directions to tangents.

In matrix notation the conic and its envelope read as a symmetric matrix and its adjugate. In the language of elliptic curves, the pairs of a point on the outer conic and a tangent of the inner one through it form a curve of genus one, a step of the path reads as adding a fixed point on it, and closing after $n$ sides as that point having order $n$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import stack
from numga import NumpyContext
from numga.algebras import PGA2D as ga
from examples.animation import save_figure, show_animation
from examples.math.poncelet import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Positions are points, intrinsic; polars are lines read by which side a point is on, extrinsic.
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
# A chord is the join of two points, a line oriented along itself; a pole is the meet of two lines.
Chord = (Point & Point).output_gatype
Pole = (Plane ^ Plane).output_gatype
Conic = ga.gatype((Plane, Point))                                              # Plane <- Point
Envelope = ga.gatype((Pole, Chord))                                            # Pole <- Chord
Motor = ga.gatype.rotor()
# Both roots of a quadratic, one per sign.
SIGNS = np.array([1.0, -1.0])
# The points at infinity along x and along y.
HEADINGS = stack([mv.x.dual(), mv.y.dual()])                                   # [2] Point

## 1. An ellipse, and a conic touching a pentagon in it

The ellipse is a sum of dyads, each coordinate line through the origin weighted by the inverse square of its semi-axis, less the line at infinity, carried to its place by a motor. Two poles make a conic held by its chords, their pair, zero on every chord through either; the pairs of opposite corners of four chords span every conic touching those four, and a fifth chord picks one. The sides of a pentagon inscribed in the ellipse fix the first inner conic; the sides of the same pentagon shrunk a little towards the centre fix the second.

In [ ]:
# The ellipse: its centre, its semi-axes, and how far its first axis is turned from x.
centre = np.array([0.4, -0.2])
semi = np.array([2.0, 1.2])
tilt = 0.5
# The corners of the pentagon, as angles around the ellipse, the next corner of each, and how far the
# second pentagon is shrunk towards the centre.
corner_angles = np.array([0.1, 0.8, 1.9, 3.0, 5.2])
following = np.array([1, 2, 3, 4, 0])
shrink = 0.97

In [ ]:
def ellipse(semi: np.ndarray, motor: Motor) -> Conic:
    """The ellipse with the given semi-axes along x and y, carried from the origin by a motor: each
    coordinate line through the origin weighted by the inverse square of its semi-axis, less the line at
    infinity."""
    axes = stack([mv.x, mv.y])                                                 # [2] Plane
    at_origin = (axes * (axes & Point) / semi**2).sum(axis=-1) - mv.w * (mv.w & Point)   # [] Plane <- Point
    return motor >> at_origin(motor << Point)                                  # [] Plane <- Point


def pair(first: Pole, second: Pole) -> Envelope:
    """Two poles as one envelope: zero on the chords through either, its pairing the product of the
    chord's pairings with the two."""
    return (first * (second & Chord) + second * (first & Chord)) / 2           # [...] Pole <- Chord


def envelope(sides: Chord) -> Envelope:
    """The envelope touching five chords on the last axis: the combination of the two pairs of
    opposite corners of the first four that vanishes on the fifth."""
    first, second, third, fourth, fifth = (sides[..., index] for index in range(5))
    corners = pair(first ^ second, third ^ fourth)                             # [...] Pole <- Chord
    diagonals = pair(first ^ third, second ^ fourth)                           # [...] Pole <- Chord
    return corners * (diagonals(fifth) & fifth) - diagonals * (corners(fifth) & fifth)   # [...] Pole <- Chord


# The ellipse turned about the origin by the tilt and carried to its centre.
placement = ((mv.xw * centre[0] + mv.yw * centre[1]) * 0.5).exp() * (mv.xy * (-tilt / 2)).exp()   # [] Motor


def on_ellipse(angles: np.ndarray, scale: float) -> Point:
    """Points at the given angles on the ellipse, drawn in towards its centre by the scale."""
    return placement >> mv.point(scale * semi * np.stack([np.cos(angles), np.sin(angles)], axis=-1))


outer = ellipse(semi, placement)                                               # [] Plane <- Point
corners = stack([on_ellipse(corner_angles, 1.0), on_ellipse(corner_angles, shrink)])   # [inners, corners] Point
inner = envelope(corners & corners[..., following])                            # [inners] Pole <- Chord

In [ ]:
print("the ellipse:\n", outer.coefficient_table())
print("the conic touching the pentagon, held by its chords:\n", inner[0].coefficient_table())

## 2. One step, twice

A chord through a vertex is the vertex joined with a direction, a point at infinity. Along a direction from a point of the ellipse, the ellipse's value is a quadratic with one root at the point, so the other crossing is rational in the conic. Its twin on tangents: among the chords through a vertex, those that touch the inner conic are the roots of a quadratic in the direction; knowing one, the incoming side, the other follows rationally from any second direction that is never a tangent, such as the direction from the ellipse's centre, which lies inside both inner conics. Only the first side from each start, with neither root known, takes a square root.

In [ ]:
def touching(inner: Envelope, vertex: Point, first: Point, second: Point) -> Scalar:
    """The envelope's pairing of the chords through a vertex along two directions: zero for one
    direction alone exactly when its chord touches the inner conic."""
    return (vertex & first) & inner(vertex & second)                           # [...] Scalar


def discriminant(inner: Envelope, points: Point) -> Scalar:
    """The discriminant of the envelope's quadratic along the directions through each point: positive
    where two tangents pass, outside the inner conic, and zero on it, where they coincide."""
    along_x, along_y = HEADINGS[0], HEADINGS[1]
    b = touching(inner, points, along_x, along_y)                              # [...] Scalar
    return b * b - touching(inner, points, along_x, along_x) * touching(inner, points, along_y, along_y)   # [...] Scalar


def tangents(inner: Envelope, points: Point) -> Point:
    """The directions of the two tangents of an envelope through each point, on the last axis: the
    roots of its quadratic along the directions, by a square root."""
    along_x, along_y = HEADINGS[0], HEADINGS[1]
    a = touching(inner, points, along_x, along_x)                              # [...] Scalar
    b = touching(inner, points, along_x, along_y)                              # [...] Scalar
    c = touching(inner, points, along_y, along_y)                              # [...] Scalar
    root = discriminant(inner, points).square_root()                           # [...] Scalar
    # The roots of `a + 2 * b * t + c * t * t` in t, as the weights of the two directions.
    return along_x * (root[..., None] * SIGNS - b[..., None]) + along_y * a[..., None]   # [..., 2] Point


def other_crossing(outer: Conic, vertex: Point, heading: Point) -> Point:
    """Where the chord from a point on the conic along a direction meets the conic again. Along
    `vertex + heading * t` the pairing is `2 * t * (outer(vertex) & heading) + t * t * (outer(heading) & heading)`,
    zero at the vertex and at the other root."""
    return vertex * (outer(heading) & heading) - heading * (2 * (outer(vertex) & heading))   # [...] Point


def other_tangent(inner: Envelope, vertex: Point, heading: Point, turned: Point) -> Point:
    """The direction of the other tangent through a vertex, given the direction of one and a second
    direction that is never a tangent: the twin of `other_crossing`, on directions. Along
    `heading + turned * t` the pairing is `2 * t * touching(heading, turned) + t * t * touching(turned, turned)`,
    zero at the known tangent and at the other root."""
    return (heading * touching(inner, vertex, turned, turned)
            - turned * (2 * touching(inner, vertex, heading, turned)))         # [...] Point


def path(outer: Conic, inner: Envelope, start: Point, heading: Point, inside: Point) -> Iterator[Point]:
    """The vertices of the path on the outer conic, from a start on it along the direction of a tangent
    of the inner conic, without end: where that chord meets the outer conic again, then the other
    tangent through the new vertex, turned against the direction from a point inside the inner conic."""
    vertex = start.broadcast_to(heading.shape)                                 # [...] Point
    while True:
        yield vertex
        vertex = other_crossing(outer, vertex, heading)                        # [...] Point
        vertex = vertex / (mv.w & vertex)                                      # [...] Point
        heading = other_tangent(inner, vertex, heading, vertex - inside)       # [...] Point
        # A direction has no weight; its length is the norm of its complement.
        heading = heading / heading.dual().norm()                              # [...] Point

## 3. Closing from every start, or from none

From starts all around the ellipse, the path of five sides about the first inner conic always closes on its start, left; about the second, which touches no pentagon inscribed in the ellipse, it always misses, right. The start is orange and the fifth vertex ringed; each inner conic is drawn where the two tangents through a point coincide.

In [ ]:
# The sides of each path; the starts around the ellipse; the frame shown as a still, and the duration of
# a frame in milliseconds.
n_sides = 5
n_frames = 120
start_angles = np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False) + 0.05
still = 10
frame_duration = 60

In [ ]:
starts = on_ellipse(start_angles, 1.0)                                         # [starts] Point
first = tangents(inner[:, None], starts[None, :])[..., 0]                      # [inners, starts] Point
# The ellipse's centre, inside both inner conics: no tangent of either passes through it.
inside = mv.point(centre)                                                      # [] Point
vertices = stack(list(islice(path(outer, inner[:, None], starts[None, :], first, inside), n_sides + 1)))   # [sides + 1, inners, starts] Point
save_figure(render.draw(outer, inner, vertices[:, :, still]), "poncelet_pentagons")
show_animation(render.animate([(outer, inner, vertices[:, :, frame]) for frame in range(n_frames)]),
               "poncelet_turn", frame_duration)

In [ ]:
# How far the fifth vertex lands from the start, about each inner conic, over all starts: the difference
# of two unit points is a direction, and its length is the norm of its complement.
gap = (vertices[n_sides] - vertices[0]).dual().norm()                          # [inners, starts] Scalar
print("largest gap about the first inner conic:", gap[0].to_array().max())
print("smallest gap about the second inner conic:", gap[1].to_array().min())

In [ ]:
# checks
# Every vertex lies on the ellipse, and every side passes through its own pole of its inner conic; the
# path about the first inner conic closes after five sides from every start, the path about the second
# does not.
np.testing.assert_allclose((outer(vertices) & vertices).to_array(), 0.0, atol=1e-12)
sides = (vertices[:-1] & vertices[1:]).normalized()                            # [sides, inners, starts] Chord
touch = (sides & inner[:, None](sides)).to_array() / np.abs(inner.kernel).max(axis=(-1, -2))[:, None]
np.testing.assert_allclose(touch, 0.0, atol=1e-10)
np.testing.assert_allclose(gap[0].to_array(), 0.0, atol=1e-10)
assert gap[1].to_array().min() > 1e-2